# Loan Risk EDA

Exploratory data analysis for the loan default risk dataset.
Run `make data` first to generate the synthetic dataset.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
df = pl.read_parquet('../data/raw/loans.parquet')
print(f'Shape: {df.shape}')
print(f'Default rate: {df["loan_default"].mean():.2%}')
df.head()

In [ ]:
# Distribution of key features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

numeric_cols = ['loan_amount', 'annual_income', 'credit_score',
                'debt_to_income_ratio', 'employment_years', 'num_delinquencies']

for ax, col in zip(axes.flat, numeric_cols):
    data = df[col].to_numpy()
    ax.hist(data, bins=40, edgecolor='black', alpha=0.7)
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Default rate by credit score band
df_pd = df.to_pandas()
bins = [300, 580, 670, 740, 800, 851]
labels = ['Poor', 'Fair', 'Good', 'Very Good', 'Exceptional']
df_pd['credit_band'] = pd.cut(df_pd['credit_score'], bins=bins, labels=labels, right=False)

import pandas as pd
default_by_band = df_pd.groupby('credit_band')['loan_default'].agg(['mean', 'count'])
default_by_band.columns = ['default_rate', 'count']
print(default_by_band)

In [ ]:
# Feature correlation with target
corr = df.select([
    'loan_amount', 'annual_income', 'credit_score',
    'debt_to_income_ratio', 'num_delinquencies', 'loan_default'
]).to_pandas().corr()['loan_default'].sort_values()

corr.drop('loan_default').plot(kind='barh', figsize=(8, 5), title='Feature Correlation with Default')
plt.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()